# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drie01/flyranks/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

Everything here runs on the **starter CSV that ships in this repo** (`data/raw/content_refresh_anonymized.csv`) — no token, no warehouse, no private data.

In [1]:
import os, sys, subprocess, re, warnings
warnings.filterwarnings("ignore")

# On Colab the repo is not on disk yet: clone it once. Locally: walk up to the repo root.
if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    if not os.path.isdir("flyranks"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/drie01/flyranks"], check=True)
    os.chdir("flyranks")
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
CSV = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(CSV), "starter CSV not found - are you inside the repo?"

import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, KFold, cross_val_predict
from sklearn.metrics import roc_auc_score
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

raw = pd.read_csv(CSV)
print("raw:", raw.shape, "| clients:", raw["client_id"].nunique())

raw: (30000, 44) | clients: 32


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**The question (plain words).** *Standing at one moment in time, can I rank pages by how likely they are to lose more than 20% of their impressions over the next 30 days?*

**The timeline (everything hangs on this).**

```
days back from export:   90 ........ 61 | 60 ........ 31 | 30 ......... 0
                         [ days 61-90 ] | [ prev_30d   ] | [ last_30d   ]
                                                         ^
                                         DECISION MOMENT (30 days before export)
   features may use ->  everything left of the caret      label is measured -> right of it
```

- **Label:** `is_declining` = last-30-day impressions are below 80% of the previous-30-day impressions (the same rule that creates `trend_direction == "down"`; the cell below checks they agree). It exists only *after* the decision moment.
- **Population:** pages with `impressions_prev_30d > 0`, because a decline cannot be measured without a baseline. This filter uses only information from *before* the decision moment, so it does not leak.
- **Features:** only the `*_prev_30d` window and things that do not change with time (keyword context, content type, word count). Age is shifted back 30 days to the decision moment.
- **Fills:** missing keyword values and word counts are filled with `0` **plus a `has_...` flag**, because the missingness follows `content_type` (a blind fill would smuggle content type in). Categoricals get an explicit `unknown` level before one-hot.

In [2]:
# --- population + label -------------------------------------------------------------------
d = raw[raw["impressions_prev_30d"] > 0].reset_index(drop=True)          # baseline exists -> decline is measurable
y = (d["impressions_last_30d"] < 0.8 * d["impressions_prev_30d"]).astype(int)
assert (y == (d["trend_direction"] == "down").astype(int)).all(), "label rule disagrees with trend_direction"
groups = d["client_id"]                                                   # context only: used to split, never a feature

# --- feature vector -------------------------------------------------------------------------
X = pd.DataFrame({
    # activity in the PREVIOUS 30 days (days 31-60 back) - already finished at the decision moment
    "log_impr_prev30":     np.log1p(d["impressions_prev_30d"]),
    "log_clicks_prev30":   np.log1p(d["clicks_prev_30d"]),
    "log_sessions_prev30": np.log1p(d["sessions_prev_30d"]),
    "ctr_prev30":          d["clicks_prev_30d"] / d["impressions_prev_30d"] * 100,   # x100 percent, like the CSV
    "has_sessions_prev30": (d["sessions_prev_30d"] > 0).astype(int),                 # 0 sessions may mean "no GA4"
    # keyword context (fixed metadata) - fill 0 + explicit flag
    "has_keyword_data":    d["search_volume"].notna().astype(int),
    "log_search_volume":   np.log1p(d["search_volume"]).fillna(0),
    "competition":         d["competition"].fillna(0),
    "cpc":                 d["cpc"].fillna(0),
    # content properties
    "has_word_count":      d["word_count"].notna().astype(int),
    "log_word_count":      np.log1p(d["word_count"]).fillna(0),
    "age_at_decision":     d["content_age_days"] - 30,                                # age 30 days ago, not today
})
cats = d[["content_type", "main_intent", "competition_level"]].fillna("unknown")
X = pd.concat([X, pd.get_dummies(cats, dtype=int)], axis=1)

assert X.notna().all().all(), "feature frame still has missing values"
assert not {"client_id", "content_id"} & set(X.columns), "an ID slipped into the features"
FEATURES = list(X.columns)
print(f"rows kept: {len(d):,} of {len(raw):,}  |  features: {len(FEATURES)}  |  base rate (is_declining): {y.mean():.3f}")
X.head(3)

rows kept: 26,612 of 30,000  |  features: 24  |  base rate (is_declining): 0.611


   log_impr_prev30  log_clicks_prev30  log_sessions_prev30  ctr_prev30  has_sessions_prev30  has_keyword_data  log_search_volume  \
0         6.895683           2.639057             2.302585    1.317123                    1                 1           2.397895   
1         8.685416           0.693147             1.098612    0.016906                    1                 1           4.510860   
2         8.714403           1.386294             1.386294    0.049269                    1                 1           0.000000   

   competition   cpc  has_word_count  log_word_count  age_at_decision  content_type_comparison article  content_type_feedly article  \
0         0.67  2.05               1        8.077758              157                                0                            0   
1         0.01  0.05               1        7.816820              415                                0                            0   
2         0.00  0.00               1        8.165079              

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing → handled how | Available when? (decision moment = 30 days before export) |
|---|---|---|---|
| `log_impr_prev30` | log of search impressions, days 31–60 back | never missing (population needs > 0) | Yes — that window had already ended before the decision moment. |
| `log_clicks_prev30` | log of search clicks, same window | never missing | Yes — same finished window. |
| `log_sessions_prev30` | log of GA4 sessions, same window | `0` when no GA4; flagged below | Yes — same finished window. |
| `ctr_prev30` | clicks ÷ impressions × 100 in that window | never missing (impressions > 0) | Yes — built only from the two finished-window columns. |
| `has_sessions_prev30` | 1 if any sessions in that window | n/a (a flag) | Yes — derived from the finished window; separates "no GA4" from "no engagement". |
| `has_keyword_data` | 1 if a target keyword is on record | n/a (a flag) | Yes — keyword assignment is page metadata, not an outcome. |
| `log_search_volume`, `competition`, `cpc` | keyword demand / difficulty / price | `0` + `has_keyword_data` flag (missing follows `content_type`) | Yes — fixed keyword context, not a traffic result. |
| `has_word_count` | 1 if word count was measured | n/a (a flag) | Yes — a measurement property of the page. |
| `log_word_count` | log of article length | `0` + `has_word_count` flag | Yes — article text length is fixed metadata. |
| `age_at_decision` | `content_age_days − 30` | never missing | Yes — the page's age 30 days ago, computed by shifting the export-day age back. |
| `content_type_*` | one-hot: keyword / feedly / comparison article | none (no blanks) | Yes — fixed at creation. |
| `main_intent_*` | one-hot search intent | blank → `unknown` level | Yes — fixed page metadata. |
| `competition_level_*` | one-hot LOW / MEDIUM / HIGH | blank → `unknown` level | Yes — fixed keyword metadata. |

The code cell below proves two things about this table: every feature in the frame is covered by a row above, and how many blanks each fill actually handled.

In [3]:
GROUPS = ["log_impr_prev30", "log_clicks_prev30", "log_sessions_prev30", "ctr_prev30", "has_sessions_prev30",
          "has_keyword_data", "log_search_volume", "competition", "cpc", "has_word_count", "log_word_count",
          "age_at_decision", "content_type_", "main_intent_", "competition_level_"]
uncovered = [c for c in FEATURES if not any(c == g or c.startswith(g) for g in GROUPS)]
assert not uncovered, f"features without a note: {uncovered}"

blanks = pd.DataFrame({
    "blank_before_fill":   d[["search_volume", "competition", "cpc", "word_count", "main_intent", "competition_level"]].isna().sum(),
})
blanks["share"] = (blanks["blank_before_fill"] / len(d)).round(3)
print("every feature has a note; blanks the fills handled:")
print(blanks.to_string())

# missingness follows content_type -> this is WHY a blind fillna(0) would be a hidden category signal
print("\nshare of blank keyword data / word_count by content_type:")
print(d.groupby("content_type")[["search_volume", "word_count"]].apply(lambda g: g.isna().mean()).round(3).to_string())

every feature has a note; blanks the fills handled:
                   blank_before_fill  share
search_volume                   1325  0.050
competition                     1325  0.050
cpc                             1325  0.050
word_count                      7445  0.280
main_intent                     1232  0.046
competition_level               1455  0.055

share of blank keyword data / word_count by content_type:
                    search_volume  word_count
content_type                                 
comparison article          0.000       0.000
feedly article              1.000       0.000
keyword article             0.013       0.299


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Three ways an answer sneaks in, and the test for each (from the leakage skill):

1. **Label-derived columns** (`trend_pct`, `trend_direction`) — the label was *computed from* them. Test: train **with** and **without** the suspect.
2. **Future / overlapping windows** (`impressions_last_30d`, and every `*_90d` total, which *contains* the last 30 days). Test: the same with/without run — and the timeline above, because a small lift does **not** make an overlapping column safe.
3. **Product flags / decisions** (`health_score`, `priority_score`, `action_type`, …). Test: scan the columns; none should exist.

Plus two honesty checks: the split must be **grouped by client** (random splits let the model memorise a client's character), and every score sits next to the **base rate**. All scores below are **out-of-fold** (each page is scored by a model that never saw its client).

### 3a. The honest number

In [4]:
def oof_proba(Xm, split="group", seed=0):
    cv = GroupKFold(n_splits=5) if split == "group" else KFold(n_splits=5, shuffle=True, random_state=seed)
    model = HistGradientBoostingClassifier(max_iter=150, random_state=0)
    return cross_val_predict(model, Xm, y, cv=cv, groups=groups if split == "group" else None,
                             method="predict_proba")[:, 1]

def oof_auc(Xm, split="group"):
    return roc_auc_score(y, oof_proba(Xm, split))

base_rate = y.mean()
p_honest = oof_proba(X, "group")
honest_auc = roc_auc_score(y, p_honest)
random_auc = oof_auc(X, "random")

top = np.argsort(-p_honest)[: int(0.10 * len(y))]            # the 10% of pages the model is most sure will decline
print(f"base rate (share declining)                : {base_rate:.3f}   (a coin-flip ranker scores AUC 0.500)")
print(f"naive baseline AUC, rank by prev impressions: {roc_auc_score(y, -d['impressions_prev_30d']):.3f}")
print(f"HONEST  AUC, split grouped by client        : {honest_auc:.3f}")
print(f"random-split AUC (pages of a client shared) : {random_auc:.3f}   <- gap = memorising clients")
print(f"precision in the model's top 10%            : {y.iloc[top].mean():.3f}  vs base rate {base_rate:.3f}")

base rate (share declining)                : 0.611   (a coin-flip ranker scores AUC 0.500)
naive baseline AUC, rank by prev impressions: 0.503
HONEST  AUC, split grouped by client        : 0.598
random-split AUC (pages of a client shared) : 0.717   <- gap = memorising clients
precision in the model's top 10%            : 0.707  vs base rate 0.611


### 3b. Suspects: train WITH each one, and see what moves

Every suspect is added to the honest frame one at a time. `lift` is the change in grouped-split AUC; a big lift is the confession.

In [5]:
def onehot(col, prefix):
    return pd.get_dummies(d[col], prefix=prefix, dtype=int)

SUSPECTS = {  # name: (type of leak, columns to add)
    "trend_pct":              ("label-derived",            d[["trend_pct"]].fillna(0)),
    "trend_direction":        ("label-derived",            onehot("trend_direction", "td")),
    "impressions_last_30d":   ("future window (the label)", d[["impressions_last_30d"]]),
    "impressions_90d":        ("window overlap (contains last 30d)", d[["impressions_90d"]]),
    "clicks_90d":             ("window overlap",           d[["clicks_90d"]]),
    "ctr":                    ("window overlap (90d rate)", d[["ctr"]]),
    "avg_position":           ("window overlap (90d mean)", d[["avg_position"]]),
    "impression_tier":        ("window overlap (tier of 90d)", onehot("impression_tier", "it")),
    "position_tier":          ("window overlap (tier of 90d)", onehot("position_tier", "pt")),
    "days_since_last_update": ("timing unknown (export-day value)", d[["days_since_last_update"]]),
    "client_code (an ID)":    ("ID / memorisation",        d["client_id"].astype("category").cat.codes.rename("client_code").to_frame()),
}

rows = []
for name, (kind, extra) in SUSPECTS.items():
    Xs = pd.concat([X, extra], axis=1)
    g, r = oof_auc(Xs, "group"), oof_auc(Xs, "random")
    rows.append({"suspect": name, "type": kind, "AUC_grouped": round(g, 3),
                 "lift": round(g - honest_auc, 3), "AUC_random": round(r, 3)})
report = pd.DataFrame(rows).sort_values("lift", ascending=False).reset_index(drop=True)
report["verdict"] = np.where(report["lift"] >= 0.05, "LEAK - big jump", "small/no jump - still excluded (see type)")
print(f"honest grouped AUC = {honest_auc:.3f}  |  honest random AUC = {random_auc:.3f}\n")
print(report.to_string(index=False))

honest grouped AUC = 0.598  |  honest random AUC = 0.717

               suspect                               type  AUC_grouped  lift  AUC_random                                   verdict
             trend_pct                      label-derived        1.000 0.402       1.000                           LEAK - big jump
       trend_direction                      label-derived        1.000 0.402       1.000                           LEAK - big jump
  impressions_last_30d          future window (the label)        0.999 0.401       0.999                           LEAK - big jump
       impressions_90d window overlap (contains last 30d)        0.679 0.081       0.791                           LEAK - big jump
          avg_position          window overlap (90d mean)        0.618 0.020       0.737 small/no jump - still excluded (see type)
       impression_tier       window overlap (tier of 90d)        0.616 0.019       0.733 small/no jump - still excluded (see type)
         position_tier   

### 3c. Product flags, private fields, and "too good" checks

- **Product flags:** the data ships observable signals only — the scan below proves no rule-output column is present.
- **Privacy:** no raw URLs, titles, queries, or client names should be anywhere in the columns or the text values.
- **Top-feature sanity:** if one feature towers over the rest, that is a symptom, not a win.

In [6]:
# --- product flags: none should exist ------------------------------------------------------
FLAG_PATTERN = re.compile(r"health_score|priority_score|action_type|needs_|is_quick_win|refresh_tier|refresh_flag|is_declining", re.I)
flag_cols = [c for c in raw.columns if FLAG_PATTERN.search(c)]
print("product-flag / label columns found in the raw CSV:", flag_cols or "none")
assert not flag_cols

# --- privacy: raw-origin fields and ID shapes ----------------------------------------------
RAW_ORIGIN = re.compile(r"url|title|query|keyword_text|domain|client_name", re.I)
bad_names = [c for c in raw.columns if RAW_ORIGIN.search(c)]
text_cols = raw.select_dtypes(include="object").columns
leaky_values = {c: int(raw[c].astype(str).str.contains(r"https?://|www\.", regex=True).sum()) for c in text_cols}
print("raw-origin column names found              :", bad_names or "none")
print("text values that look like a URL           :", {k: v for k, v in leaky_values.items() if v} or "none")
print("ids are pseudonyms (shape check)           :",
      raw["client_id"].str.match(r"^client_[0-9a-f]{10}$").all() and raw["content_id"].str.match(r"^content_[0-9a-f]{12}$").all())
print("IDs inside the feature frame               :", sorted({"client_id", "content_id"} & set(FEATURES)) or "none")

# --- is any single feature towering? permutation importance on clients the model never saw --
from sklearn.inspection import permutation_importance
tr, te = next(GroupKFold(n_splits=5).split(X, y, groups))
m = HistGradientBoostingClassifier(max_iter=150, random_state=0).fit(X.iloc[tr], y.iloc[tr])
imp = permutation_importance(m, X.iloc[te], y.iloc[te], scoring="roc_auc", n_repeats=5, random_state=0)
top_feats = pd.Series(imp.importances_mean, index=FEATURES).sort_values(ascending=False).head(6).round(4)
print("\ntop features by drop in held-out AUC when shuffled (a tower here would be a red flag):")
print(top_feats.to_string())

# the leader is age_at_decision: legitimate metadata (not label-derived) - but does the score depend on it?
no_age = [c for c in FEATURES if c != "age_at_decision"]
print(f"\nhonest AUC with age_at_decision   : {honest_auc:.3f}")
print(f"honest AUC without age_at_decision: {oof_auc(X[no_age], 'group'):.3f}")

product-flag / label columns found in the raw CSV: none
raw-origin column names found              : none
text values that look like a URL           : none
ids are pseudonyms (shape check)           : True
IDs inside the feature frame               : none

top features by drop in held-out AUC when shuffled (a tower here would be a red flag):
age_at_decision        0.0751
log_impr_prev30        0.0117
log_sessions_prev30    0.0073
log_search_volume      0.0013
log_clicks_prev30      0.0005
ctr_prev30             0.0005

honest AUC with age_at_decision   : 0.598
honest AUC without age_at_decision: 0.581


**Reading 3c.** No product flags, no raw-origin fields, no IDs in the features. One honest feature leads — `age_at_decision` — and it is worth saying why that is *not* a leak: it is fixed page metadata (the page's age 30 days ago), it is not computed from the label, and it still carries signal on clients the model never saw. But it dominates, so the honest score leans on it: the without-age number above shows how much. I report it as **observed and directional** and do not claim age *causes* declines.

### 3d. Delete the leak, keep the honest number

Last step of the lesson: build the leaky version on purpose, show the jump, then show the honest frame is untouched and its score is the one I keep.

In [7]:
X_leak = X.assign(leak_trend_pct=d["trend_pct"].fillna(0))     # a label-derived column, added on purpose
leak_auc = oof_auc(X_leak, "group")
print(f"WITH the leak    : AUC {leak_auc:.3f}   <- looks perfect, means nothing (answer was in the input)")

del X_leak                                                      # delete it
assert "leak_trend_pct" not in X.columns
assert not any(c.startswith("trend_") for c in X.columns)
assert leak_auc > honest_auc + 0.2, "the test harness failed to detect an obvious leak"
print(f"WITHOUT the leak : AUC {oof_auc(X, 'group'):.3f}   <- the number I keep (grouped by client, out-of-fold)")

WITH the leak    : AUC 1.000   <- looks perfect, means nothing (answer was in the input)
WITHOUT the leak : AUC 0.598   <- the number I keep (grouped by client, out-of-fold)


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

**One named limitation:** the starter CSV is a single snapshot with 90-day totals, so I could not build a true time split and had to give up the overlapping position and CTR totals. That removes real signal (position is a classic ranking driver), which is part of why the honest score is only modest. Treat the result as **observed and directional**, not as proof of what drives a decline.

In [8]:
EXCLUDED = {
    "trend_direction":        "the label is computed from it (label-derived)",
    "trend_pct":              "the label's source column (label-derived)",
    "impressions_last_30d":   "it IS the label window - the outcome itself",
    "clicks_last_30d":        "inside the label window (future relative to the decision)",
    "sessions_last_30d":      "inside the label window (future relative to the decision)",
    "impressions_90d":        "90-day total contains the last 30 days (window overlap)",
    "clicks_90d / sessions_90d / users_90d / pageviews_90d / engaged_sessions_90d / scroll_events_90d / ai_sessions_90d":
                              "all 90-day totals contain the last 30 days (window overlap)",
    "days_with_impressions / days_with_sessions": "counted over the full 90 days (window overlap)",
    "ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct": "ratios of the overlapping 90-day totals",
    "impression_tier, position_tier": "buckets of overlapping 90-day totals",
    "days_since_last_update / freshness_tier": "export-day value: cannot tell if the update came before or after the decision",
    "content_age_days / age_tier / age_tier_order": "replaced by age_at_decision (age 30 days earlier)",
    "word_count / char_count tiers": "redundant with log_word_count",
    "provider_used, model_used": "not a decision-time signal and mostly blank (data dictionary says not a model feature)",
    "client_id, content_id":  "pseudonymous IDs: used only to group the split, never to learn from",
}
excluded = pd.DataFrame({"field(s) refused": list(EXCLUDED), "why": list(EXCLUDED.values())})
print(excluded.to_string(index=False))

# machine check: nothing on the refusal list is in the feature frame
refused_cols = {"trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                "impressions_90d", "clicks_90d", "sessions_90d", "users_90d", "pageviews_90d", "engaged_sessions_90d",
                "scroll_events_90d", "ai_sessions_90d", "days_with_impressions", "days_with_sessions", "ctr",
                "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct", "impression_tier", "position_tier",
                "days_since_last_update", "freshness_tier", "content_age_days", "age_tier", "age_tier_order",
                "word_count", "char_count", "provider_used", "model_used", "client_id", "content_id"}
assert not refused_cols & set(FEATURES), refused_cols & set(FEATURES)
print("\nOK: none of the refused fields is in the feature frame.")

                                                                                                  field(s) refused                                                                                    why
                                                                                                   trend_direction                                          the label is computed from it (label-derived)
                                                                                                         trend_pct                                              the label's source column (label-derived)
                                                                                              impressions_last_30d                                            it IS the label window - the outcome itself
                                                                                                   clicks_last_30d                              inside the label window (future relative to the 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.